# Phase 1: MoCA and Entorhinal MRI Measures in Older NACC Participants

This notebook reproduces the UDS/MRI linkage, participant counts, data-quality checks, descriptive summaries, and figures. Older adults are operationally defined as age 65 or older. Each supplied file has one row per NACCID, but there are no visit IDs or dates; therefore, the retained record cannot be verified as a true baseline or as temporally aligned with MRI. This phase describes unadjusted relationships. Age- and education-adjusted estimates belong to the later modeling phase.

In [ ]:
from pathlib import Path
import json, subprocess, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "udslong_imputed.csv").exists())
subprocess.run([sys.executable, str(ROOT / "src" / "phase1_analysis.py")], check=True, cwd=ROOT)
results = json.loads((ROOT / "reports/phase1/phase1_results.json").read_text(encoding="utf-8"))
print(json.dumps(results["linkage"], indent=2))

## Unit of analysis and linkage rule

The unit is one unique participant (`NACCID`). Each supplied CSV has 1,237 rows and 1,237 unique IDs, with no duplicates or unmatched IDs. Retain the single supplied row per participant, then restrict the analytic sample to `NACCAGE >= 65`. Since visit IDs/dates and scan dates are absent, this is an available-record rule rather than a verified baseline selection.

In [ ]:
print("Design:", json.dumps(results["design"], indent=2))
print("Linkage:", json.dumps(results["linkage"], indent=2))

## Descriptive statistics and missingness

SD is the sample standard deviation. IQR is Q3 minus Q1 using linearly interpolated quartiles. Skewness is the adjusted sample skewness. `EDUC=99` is treated as unknown; MOCATOTS values outside 0–30 are invalid. Nonpositive entorhinal volumes are not valid measurements. The UDS file is imputed/processed, but its method and original missingness mask were not supplied.

In [ ]:
for name, s in results["summary_statistics"].items():
    print(f"{name}: n={s['n']}, mean={s['mean']:.3f}, median={s['median']:.3f}, SD={s['sd']:.3f}, IQR={s['iqr']:.3f}, skew={s['skewness']:.3f}")
print("\nMissingness:")
for row in results["missingness"]: print(row)

## Figures and interpretation

The output includes a MOCATOTS histogram, a MoCA-versus-bilateral-entorhinal-volume scatterplot with a simple linear trend, a MoCA boxplot by cognitive-status group, and a Pearson correlation heatmap. The trend and correlations are descriptive and unadjusted; they do not control for age or education. For the primary question, the next phase should fit separate multiple linear regression models for entorhinal volume and thickness, adjusting for `NACCAGE` and `EDUC`, and assess model assumptions.

In [ ]:
for name in results["figure_files"]:
    print(name)
print("\nPearson correlation matrix:")
print(json.dumps(results["pearson_correlation_matrix"], indent=2))

## Cautious interpretation

The analytic sample contains 864 participants aged 65 or older; 855 have complete data across all core variables. Pairwise correlations may use slightly different sample sizes. Results apply to this available NACC clinical/research subset, not a probability sample of the general population. Observational associations do not establish causation. Check missingness, imputation effects, MRI quality, and UDS–MRI timing before adjusted-model interpretation.